# Passo 6: modelo final para o app

O modelo foi avaliado no passo 5 (treino até jun/2024, teste em 2025). Agora ele é **treinado de novo com todas as linhas que já têm alvo** (jul/2022 a jun/2025), para fazer as previsões mais atualizadas possíveis.

As linhas de **jul a dez/2025** têm todas as variáveis, mas ainda não têm alvo (o futuro delas é 2026). São essas que o app vai pontuar.

Arquivos gerados em `app/` (entram no Git, para o app funcionar sem baixar os dados do BCB):
- `modelo.joblib`: o Pipeline treinado (preparação + floresta aleatória).
- `dados_app.parquet`: histórico dos recortes com as variáveis.

In [1]:
from pathlib import Path

import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src.dados import PASTA_PROCESSED
from src.variaveis import VARIAVEIS, VARIAVEIS_CAT, VARIAVEIS_NUM, criar_variaveis

# Painel do passo 3 (com alvo) + variáveis calculadas pela função de src/variaveis.py
painel = criar_variaveis(pd.read_parquet(PASTA_PROCESSED / "painel_alvo.parquet"))

# Linhas com todas as variáveis (saem os 6 primeiros meses de cada recorte)
completas = painel.dropna(subset=VARIAVEIS_NUM)
com_alvo = completas.dropna(subset=["alvo"])
sem_alvo = completas[completas["alvo"].isna()]

print(f"Com alvo (treino final): {len(com_alvo)} linhas, "
      f"{com_alvo['data_base'].min():%m/%Y} a {com_alvo['data_base'].max():%m/%Y}, "
      f"{100 * com_alvo['alvo'].astype(float).mean():.1f}% positivos")
print(f"Sem alvo (para o app pontuar): {len(sem_alvo)} linhas, "
      f"{sem_alvo['data_base'].min():%m/%Y} a {sem_alvo['data_base'].max():%m/%Y}")

Com alvo (treino final): 13896 linhas, 07/2022 a 06/2025, 16.8% positivos
Sem alvo (para o app pontuar): 2316 linhas, 07/2025 a 12/2025


In [2]:
# Conferência: a função de src/variaveis.py precisa gerar exatamente
# as mesmas variáveis do notebook 04 (salvas em base_modelo.parquet)
base_04 = pd.read_parquet(PASTA_PROCESSED / "base_modelo.parquet")
pd.testing.assert_frame_equal(
    com_alvo[["data_base"] + VARIAVEIS].reset_index(drop=True),
    base_04[["data_base"] + VARIAVEIS].reset_index(drop=True),
    check_dtype=False,
)
print("Variáveis idênticas às do passo 4.")

Variáveis idênticas às do passo 4.


In [3]:
# Mesmo Pipeline e mesmos parâmetros da floresta escolhida no passo 5
modelo = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", VARIAVEIS_NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), VARIAVEIS_CAT),
    ])),
    ("modelo", RandomForestClassifier(
        n_estimators=300,
        min_samples_leaf=20,
        class_weight="balanced",
        n_jobs=-1,
        random_state=42,
    )),
])

# Agora o fit usa todas as linhas com alvo (jul/2022 a jun/2025)
modelo.fit(com_alvo[VARIAVEIS], com_alvo["alvo"].astype(int))
print("Modelo treinado.")

Modelo treinado.


In [4]:
PASTA_APP = Path("app")
PASTA_APP.mkdir(exist_ok=True)

# joblib salva o Pipeline inteiro num arquivo; compress=3 reduz o tamanho
joblib.dump(modelo, PASTA_APP / "modelo.joblib", compress=3)

# Dados do app: só as colunas que ele usa
colunas = ["data_base"] + VARIAVEIS_CAT + ["carteira_ativa"] + VARIAVEIS_NUM + ["alvo"]
painel[colunas].to_parquet(PASTA_APP / "dados_app.parquet", index=False)

for arquivo in ("modelo.joblib", "dados_app.parquet"):
    tamanho = (PASTA_APP / arquivo).stat().st_size / 1e6
    print(f"{arquivo}: {tamanho:.1f} MB")

modelo.joblib: 3.1 MB
dados_app.parquet: 1.5 MB


In [5]:
# Prévia do que o app vai mostrar: os 10 recortes com maior nota de risco no último mês.
# A nota (0 a 1) serve para ORDENAR os recortes; não é uma probabilidade calibrada.
ultimo_mes = sem_alvo[sem_alvo["data_base"] == sem_alvo["data_base"].max()].copy()
ultimo_mes["nota_risco"] = modelo.predict_proba(ultimo_mes[VARIAVEIS])[:, 1]

print(f"Mês-base: {ultimo_mes['data_base'].max():%m/%Y} ({len(ultimo_mes)} recortes)")
(
    ultimo_mes.sort_values("nota_risco", ascending=False)
    [["uf", "cliente", "modalidade", "taxa_inad", "taxa_atraso_15_90", "nota_risco"]]
    .head(10)
    .round(3)
)

Mês-base: 12/2025 (386 recortes)


,uf,cliente,modalidade,taxa_inad,taxa_atraso_15_90,nota_risco
1247,AM,PF,Financiamentos rurais (ex-financiamentos rura...,3.182,1.503,0.949
14591,RR,PF,Financiamentos rurais (ex-financiamentos rura...,3.414,1.689,0.945
4559,ES,PF,Financiamentos rurais (ex-financiamentos rura...,1.647,1.456,0.906
11855,PR,PF,Financiamentos rurais (ex-financiamentos rura...,4.553,1.011,0.873
14975,RS,PF,Financiamentos rurais (ex-financiamentos rura...,4.872,1.189,0.871
17087,SP,PF,Financiamentos rurais (ex-financiamentos rura...,4.412,1.262,0.869
3839,DF,PF,Financiamentos rurais (ex-financiamentos rura...,3.439,1.306,0.864
12671,RJ,PF,Financiamentos rurais (ex-financiamentos rura...,4.081,0.978,0.859
15743,SC,PF,Financiamentos rurais (ex-financiamentos rura...,2.504,0.552,0.852
191,AC,PF,Financiamentos rurais (ex-financiamentos rura...,5.000,1.579,0.847
